# Clase 189 — DoubleML / EconML: ML para causalidad

El problema: con confounding observado high-dim, regresión naive sesga. **Double/Debiased ML** (Chernozhukov 2018) usa ML para residualizar Y y T, y aplica Frisch-Waugh-Lovell sobre los residuos → estimador eficiente con CI válido.
Requiere: `pip install numpy scikit-learn` (opcional `doubleml`).

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(42)
n, p = 2000, 20
X = rng.normal(0, 1, (n, p))
beta = rng.normal(0, 1, p)
# Tratamiento depende de X (confounding)
logit = X @ beta * 0.3
T = (rng.uniform(0, 1, n) < 1/(1+np.exp(-logit))).astype(float)
# Outcome depende de X y T; ATE verdadero = 2.0
TRUE_ATE = 2.0
Y = X @ beta + TRUE_ATE * T + rng.normal(0, 1, n)
print(f'n={n}  p={p}  P(T=1)={T.mean():.3f}  TRUE ATE = {TRUE_ATE}')

## Naive: diferencia de medias (sesgado)

In [ ]:
naive = Y[T==1].mean() - Y[T==0].mean()
print(f'Naive diff-in-means = {naive:.3f}  (TRUE = {TRUE_ATE}) -> sesgado por confounding via X')

## Regresión OLS sobre X y T (no DoubleML)

In [ ]:
from numpy.linalg import lstsq
Xfull = np.column_stack([T, X, np.ones(n)])
coef, *_ = lstsq(Xfull, Y, rcond=None)
print(f'OLS coef sobre T = {coef[0]:.3f}  (mejora cuando el modelo es lineal correcto)')

## DoubleML manual — cross-fitting K=2
1. Split en K folds. En cada fold-fuera, entrenar:
   - $\hat g(X) \approx E[Y|X]$
   - $\hat m(X) \approx E[T|X]$ (propensity)
2. Residualizar: $\tilde Y = Y - \hat g(X)$, $\tilde T = T - \hat m(X)$.
3. $\hat\theta = \dfrac{\sum \tilde T \tilde Y}{\sum \tilde T^2}$ (Frisch-Waugh-Lovell).

In [ ]:
def doubleml_plr(X, T, Y, K=2, seed=42):
    n = len(Y)
    Y_res = np.zeros(n); T_res = np.zeros(n)
    kf = KFold(n_splits=K, shuffle=True, random_state=seed)
    for train_idx, test_idx in kf.split(X):
        # nuisance Y: Ridge
        g = Ridge(alpha=1.0).fit(X[train_idx], Y[train_idx])
        # nuisance T: LogReg (T binario)
        m = LogisticRegression(max_iter=1000, C=1.0).fit(X[train_idx], T[train_idx])
        Y_res[test_idx] = Y[test_idx] - g.predict(X[test_idx])
        T_res[test_idx] = T[test_idx] - m.predict_proba(X[test_idx])[:, 1]
    # FWL
    theta = (T_res * Y_res).sum() / (T_res**2).sum()
    # SE robusto (sandwich simplificado)
    psi = T_res * (Y_res - theta * T_res)
    J = (T_res**2).mean()
    var = (psi**2).mean() / (J**2) / n
    se = np.sqrt(var)
    return theta, se

theta_hat, se = doubleml_plr(X, T, Y, K=2)
ci = (theta_hat - 1.96*se, theta_hat + 1.96*se)
print(f'DoubleML manual: ATE = {theta_hat:.3f}  SE = {se:.4f}  CI95 = [{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'TRUE ATE       : {TRUE_ATE}  -> dentro del CI: {ci[0] <= TRUE_ATE <= ci[1]}')

## Con más folds (K=5)

In [ ]:
for K in [2, 3, 5, 10]:
    t, s = doubleml_plr(X, T, Y, K=K)
    print(f'K={K:2d}  ATE={t:.3f}  SE={s:.4f}')

## DoubleML library (opcional)

In [ ]:
try:
    import doubleml as dml
    import pandas as pd
    df = pd.DataFrame(X, columns=[f'x{i}' for i in range(p)])
    df['Y'] = Y; df['T'] = T
    data = dml.DoubleMLData(df, y_col='Y', d_cols='T', x_cols=[f'x{i}' for i in range(p)])
    ml_g = Ridge(alpha=1.0)
    ml_m = LogisticRegression(max_iter=1000)
    plr = dml.DoubleMLPLR(data, ml_g, ml_m, n_folds=2)
    plr.fit()
    print('DoubleMLPLR:')
    print(plr.summary)
except ImportError:
    print('doubleml no instalado; usar `pip install doubleml` para validar.')
    print(f'Manual da: ATE={theta_hat:.3f}, SE={se:.4f}')

## ¿Por qué cross-fitting?
Sin cross-fitting (entrenar y predecir sobre los mismos datos), $\hat g$ y $\hat m$ **overfittean** → residuos correlacionan con T → sesgo. Cross-fitting rompe esa correlación → CI válido.

In [ ]:
# Demo: SIN cross-fitting (in-sample)
g = Ridge(alpha=1.0).fit(X, Y)
m = LogisticRegression(max_iter=1000).fit(X, T)
Y_res_is = Y - g.predict(X)
T_res_is = T - m.predict_proba(X)[:, 1]
theta_is = (T_res_is * Y_res_is).sum() / (T_res_is**2).sum()
print(f'IN-sample (sin cross-fit): ATE = {theta_is:.3f}  -> sesgado, no usar')
print(f'Cross-fitting K=5       : ATE = {doubleml_plr(X,T,Y,K=5)[0]:.3f}')

## Takeaways
1. Diff-of-means sin ajuste = **sesgado** con confounding.
2. **DoubleML** = ML flexible (Ridge/RF/XGB) + cross-fitting + FWL → ATE consistente y normal asintóticamente.
3. Cross-fitting es **crítico**: evita el sesgo por overfitting de las nuisance functions.
4. Librerías de producción: `doubleml`, `econml` (Microsoft). Soportan CATE, IV, panel.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

### Ejercicio 1 — Dataset sintético
`Y = 2·T + 3·X1 + 1.5·X2² + eps` con propensión no lineal en X2. ATE verdadero = 2.

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
rng = np.random.default_rng(42)
n = 4000
X1 = rng.normal(0, 1, n); X2 = rng.normal(0, 1, n)
p = 1/(1 + np.exp(-(0.6*X1 + 0.9*(X2**2 - 1.0))))   # propension no lineal en X2
T = rng.binomial(1, p)
Y = 2.0*T + 3.0*X1 + 1.5*X2**2 + rng.normal(0, 1, n)  # ATE verdadero = 2
X = np.column_stack([X1, X2])
print(f"ATE verdadero = 2 | n={n}, P(T=1)={T.mean():.2f}")
assert 0.3 < T.mean() < 0.7

### Ejercicio 2 — DML básico (cross-fitting K=5)
DML partialling-out con Random Forest como nuisance. Reportar θ̂ e IC95%.

In [ ]:
def dml_plr(X, T, Y, n_folds=5, seed=42):
    kf = KFold(n_folds, shuffle=True, random_state=seed)
    Tr = np.zeros(len(T)); Yr = np.zeros(len(Y))
    for tr, te in kf.split(X):
        g = RandomForestRegressor(200, random_state=seed).fit(X[tr], Y[tr])
        m = RandomForestRegressor(200, random_state=seed).fit(X[tr], T[tr])
        Yr[te] = Y[te] - g.predict(X[te])
        Tr[te] = T[te] - m.predict(X[te])
    theta = (Tr @ Yr) / (Tr @ Tr)
    se = np.sqrt(np.mean((Yr - theta*Tr)**2) / (len(T) * np.mean(Tr**2)))
    return theta, se

theta, se = dml_plr(X, T, Y)
print(f"DML theta = {theta:.3f}  IC95% = ({theta-1.96*se:.3f}, {theta+1.96*se:.3f})")
assert abs(theta - 2.0) < 0.25

### Ejercicio 3 — OLS vs DML
OLS lineal `Y~T+X1+X2` queda sesgado porque el confounding pasa por `X2²`. DML lo recupera.

In [ ]:
b_ols = sm.OLS(Y, sm.add_constant(np.column_stack([T, X1, X2]))).fit().params[1]
naive = Y[T==1].mean() - Y[T==0].mean()
print(f"diferencia ingenua           = {naive:.3f}")
print(f"OLS lineal (sesgado por X2^2)= {b_ols:.3f}")
print(f"DML RF                        = {theta:.3f}  (verdadero 2)")
assert abs(theta - 2.0) < 0.3

### Ejercicio 4 — CATE con causal forest (T-learner)
`econml` no está instalada → CATE por T-learner con RF sobre datos aleatorizados. Efecto heterogéneo `τ(x)=1+X1`; el CATE estimado crece con X1.

In [ ]:
rng = np.random.default_rng(0)
nh = 6000
Xh = rng.normal(0, 1, (nh, 3))
Th = rng.binomial(1, 0.5, nh)                 # aleatorizado -> CATE limpio
tau = 1.0 + 1.0*Xh[:, 0]                        # efecto heterogeneo en X1
Yh = tau*Th + Xh[:, 1] + rng.normal(0, 1, nh)
m1 = RandomForestRegressor(200, random_state=0).fit(Xh[Th==1], Yh[Th==1])
m0 = RandomForestRegressor(200, random_state=0).fit(Xh[Th==0], Yh[Th==0])
cate = m1.predict(Xh) - m0.predict(Xh)          # T-learner
qs = np.quantile(Xh[:, 0], [0.25, 0.5, 0.75])
bins = np.digitize(Xh[:, 0], qs)
print("CATE medio por cuartil de X1:")
for b in range(4):
    print(f"  cuartil {b}: X1~{Xh[bins==b,0].mean():+.2f}  CATE={cate[bins==b].mean():+.2f}")
corr = np.corrcoef(Xh[:, 0], cate)[0, 1]
print(f"corr(X1, CATE estimado) = {corr:.3f}  (verdadero: crece con X1)")
assert corr > 0.5

### Ejercicio 5 — Policy tree
`econml.policy` no está instalada → política = tratar donde el CATE estimado > 0, resumida en un árbol interpretable. Tratar selectivamente supera a tratar a todos cuando hay unidades con efecto negativo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
treat_policy = (cate > 0).astype(int)                         # umbral de costo = 0
pol = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xh, treat_policy)
val_policy = (tau * treat_policy).mean()                       # valor esperado de la politica
val_all = tau.mean()                                           # tratar a todos
print(f"valor politica (CATE>0) = {val_policy:.3f}")
print(f"valor tratar a todos    = {val_all:.3f}")
print(f"valor no tratar a nadie = 0.000")
print(f"arbol (prof=3) accuracy sobre la politica = {pol.score(Xh, treat_policy):.2f}")
assert val_policy >= val_all - 1e-9    # evita las unidades con tau<0